**Live Deployment Link:** https://PASTE-YOUR-STREAMLIT-LINK-HERE.streamlit.app/

---

# Global Air Pollution: AQI Prediction System, 10-Fold Cross Validation (Full Dataset)

## Setup: Running This Notebook on Google Colab

No installation is needed. Google Colab already has every library this notebook uses.

1. Go to **https://colab.research.google.com** and sign in with a Google account.
2. Click **File > Upload notebook** and choose this `.ipynb` file.
3. Click **Runtime > Run all**.
4. When **Step 2.1** runs, a **"Choose Files"** button appears below that cell. Click it and select **`global_air_pollution_dataset.csv`** from your computer. The notebook continues automatically after the upload.

> **Note:** Colab deletes uploaded files when the runtime is reset or disconnected. If you get a *"file not found"* error, run Step 2.1 again and re-upload `global_air_pollution_dataset.csv`.
>
> **Run time:** the whole notebook (two cross-validation experiments of 5 algorithms x 10 folds) takes roughly 10 to 20 minutes in Colab. The Support Vector Regression and Extra Trees algorithms are the slowest.

---

## Introduction

##### **Definition** A regression ML task that predicts the **Air Quality Index (AQI)** of a city from the AQI values of three of its pollutants (carbon monoxide, ozone and nitrogen dioxide) and its country, **without using the fine-particle (PM2.5) reading**.
##### **Application** Public air-quality dashboards, health warnings for sensitive groups, city and environmental planning, and **estimating the overall AQI of a city when its PM2.5 sensor is missing or broken** (a common real-world situation).

### Aim of This Notebook

> **Which of 5 regression algorithms predicts the AQI of a city best, and how sure can we be about that answer?**

To answer it fairly we use **K-Fold Cross Validation** and compare the algorithms using **Root Mean Squared Error (RMSE)**. This notebook uses the **full dataset** with **10-fold cross validation**, in the same way as the Car Price notebook, but with five **different** algorithms: Ridge Regression, Lasso Regression, Support Vector Regression (SVR), Extra Trees and Histogram Gradient Boosting.

**Why PM2.5 is left out of the main experiment:** by definition the overall AQI is the *highest* of the individual pollutant AQI values, and PM2.5 is the highest one in about 78% of cities (Step 3.3). With PM2.5 as an input the problem becomes almost *"pick the largest number"*, which is too easy to be a meaningful comparison of learning algorithms. Without it, the algorithms must *really learn* to estimate the AQI from weaker clues (CO, Ozone, NO2 and the country). At the end (Step 13) we also run the full experiment **with** PM2.5 as a short extra experiment, to show the difference.

### What is K-Fold Cross Validation?

A single train/test split can be lucky or unlucky: the result depends on *which* cities happened to land in the test set. K-Fold Cross Validation removes this luck:

1. Shuffle the data and split it into **K equal parts (folds)**.
2. Train the model on **K-1 folds** and test it on the **remaining fold**.
3. Repeat **K times**, so that **every fold is used as the test set exactly once**.
4. Report the **average** score over the K rounds (and the standard deviation, to show how much it varies).

With **K = 10** and about **23,000 cities**, each test fold has about 2,300 cities and each training set has about 20,700 cities:

```
Round  1:  [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  2:  [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  3:  [train] | [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  4:  [train] | [train] | [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train]
Round  5:  [train] | [train] | [train] | [train] | [TEST ] | [train] | [train] | [train] | [train] | [train]
Round  6:  [train] | [train] | [train] | [train] | [train] | [TEST ] | [train] | [train] | [train] | [train]
Round  7:  [train] | [train] | [train] | [train] | [train] | [train] | [TEST ] | [train] | [train] | [train]
Round  8:  [train] | [train] | [train] | [train] | [train] | [train] | [train] | [TEST ] | [train] | [train]
Round  9:  [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [TEST ] | [train]
Round 10:  [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [TEST ]
```

Every city is used for testing **exactly once** and for training **K-1 times**.

### Evaluation Measure: Root Mean Squared Error (RMSE)

```
RMSE = square root of ( (1/n) x sum of (actual AQI - predicted AQI)^2 )
```

- *actual AQI* is the true AQI of a city, *predicted AQI* is the model's prediction, and *n* is the number of test cities.
- RMSE is in the **same unit as the output (AQI points)**, so it reads as *"on average, predictions are off by about X AQI points"*.
- **Lower RMSE = better model.** Because errors are squared, RMSE penalises **large mistakes** heavily.

Two extra measures are reported next to RMSE: **MAE** (Mean Absolute Error, the plain average size of the error) and **R2** (the share of the variation in AQI that the model explains; 1.0 is perfect).

## Input and Output

### Input (Features / Attributes)

| Feature | Description | Type | Possible Values |
|---|---|---|---|
| **CO AQI Value** | AQI value of carbon monoxide | Numerical | 0 - 133 |
| **Ozone AQI Value** | AQI value of ozone | Numerical | 0 - 235 |
| **NO2 AQI Value** | AQI value of nitrogen dioxide | Numerical | 0 - 91 |
| **Country** | Country of the city | Categorical | 175 countries (India, Brazil, Germany, ...) |

> **PM2.5 AQI Value is deliberately not an input** (reason in Step 3.3). It stays in the dataset, and Step 13 shows what happens when it is added.

### Output (Prediction)

| Output Attribute | Description | Data Type |
|---|---|---|
| **AQI Value** | Overall Air Quality Index of the city (higher = more polluted) | Numerical (6 - 500) |

### AQI Categories (for reference)

| AQI Value | Category |
|---|---|
| 0 - 50 | Good |
| 51 - 100 | Moderate |
| 101 - 150 | Unhealthy for Sensitive Groups |
| 151 - 200 | Unhealthy |
| 201 - 300 | Very Unhealthy |
| 301 - 500 | Hazardous |

### Example Records

| Country | CO AQI | Ozone AQI | NO2 AQI | PM2.5 AQI (shown only, NOT used) | AQI Value |
|---|---|---|---|---|---|
| Russian Federation | 1 | 36 | 0 | 51 | 51 |
| Brazil | 1 | 5 | 1 | 41 | 41 |
| Italy | 1 | 39 | 2 | 66 | 66 |
| Poland | 1 | 34 | 0 | 20 | 34 |
| France | 0 | 22 | 0 | 6 | 22 |

## Developer & System Information

| **Attribute** | **Details** |
|---|---|
| **Student** | Haziq Mahmood (FA24-BSE-049) |
| **Course** | AIC354 Machine Learning Fundamentals, Assignment 3 |
| **Instructor** | Dr. Rao Muhammad Adeel Nawab |
| **Institution** | COMSATS University Islamabad, Lahore Campus (Fall 2026) |
| **Program Name** | `aqi_prediction_kfold_full_dataset_10fold` |
| **IDE** | Google Colab |
| **Programming Language** | Python 3 |
| **Libraries** | NumPy, Pandas, scikit-learn, Matplotlib, tqdm, Pickle (built-in). Exact versions are printed in Step 1. |
| **Dataset** | `global_air_pollution_dataset.csv` (23,463 cities, 12 attributes; 23,035 cities after removing incomplete rows) |

### Table of Contents
- **Step 1:** Import Libraries
- **Step 2:** Load Data
  - **Step 2.1:** Upload the Dataset (Google Colab)
  - **Step 2.2:** Read the Dataset
- **Step 3:** Understand and Pre-process the Data
  - **Step 3.1:** Understand the Data
  - **Step 3.2:** A Closer Look at the Output (AQI)
  - **Step 3.3:** How Is the Overall AQI Related to the Pollutant Values? (and why PM2.5 is left out)
  - **Step 3.4:** Pre-process the Data
- **Step 4:** Feature Extraction
- **Step 5:** Build the Pre-processing Pipeline
- **Step 6:** Define the 5 Regression Algorithms
- **Step 7:** Set Up 10-Fold Cross Validation
- **Step 8:** Run 10-Fold Cross Validation for All 5 Algorithms
- **Step 9:** Compare the Algorithms (including **Step 9.5:** Do the Algorithms Beat Simple Rules?)
- **Step 10:** Train the Final Model
- **Step 11:** Execute the Application Phase
- **Step 12:** Execute the Feedback Phase
- **Step 13:** Extra Experiment: The Same Comparison WITH PM2.5
- **Conclusion**

# Step 1: Import Libraries

In [ ]:
import os
import json
import pickle
import time

import numpy as np
import pandas as pd

import sklearn
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- The 5 Regression Algorithms (none of them is used in the Car Price notebooks) ---
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor

# --- Charts and progress bar ---
import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"Python libraries loaded. pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | matplotlib {matplotlib.__version__}")

# Step 2: Load Data
## Step 2.1: Upload the Dataset (Google Colab)

In Google Colab, run the cell below and click **Choose Files** to upload `global_air_pollution_dataset.csv`. If you run the notebook on your own computer, keep the CSV file in the same folder as the notebook and this cell does nothing.

In [ ]:
DATA_FILE = "global_air_pollution_dataset.csv"

# google.colab is only available inside Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(DATA_FILE):
    print(f"Please click 'Choose Files' and select '{DATA_FILE}' ...")
    uploaded = files.upload()

    # If the browser renamed the file (e.g. 'global_air_pollution_dataset (1).csv'), rename it back
    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != DATA_FILE:
        os.rename(uploaded_name, DATA_FILE)

if os.path.exists(DATA_FILE):
    print(f"'{DATA_FILE}' is ready to use.")
else:
    raise FileNotFoundError(f"'{DATA_FILE}' not found. Please run this cell again and upload it.")

## Step 2.2: Read the Dataset

This notebook uses **all** the cities in the dataset.

In [ ]:
# Load the full global air pollution dataset
raw = pd.read_csv(DATA_FILE)

print(f"Loaded dataset: {raw.shape[0]:,} rows and {raw.shape[1]} attributes")

# Preview the dataset
raw.head()

# Step 3: Understand and Pre-process the Data
## Step 3.1: Understand the Data

In [ ]:
"""
Purpose of this section:
-------------------------
1. Print the names of all attributes (columns).
2. Show the total number of instances (rows).
3. Show how many values are missing in each attribute.
4. Show summary statistics of the numerical attributes.
"""

print("\nAttributes in the Dataset:")
print("==========================\n")
print(list(raw.columns))

print("\nNumber of Instances in the Dataset:")
print("===================================\n")
print(len(raw))

print("\nMissing Values in Each Attribute:")
print("=================================\n")
missing = raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

print("\nSummary Statistics of the Numerical Attributes:")
print("===============================================\n")
raw.describe().T.round(2)

**What we see:**

- The dataset has **23,463 cities** and **12 attributes**.
- Only two attributes have missing values: **Country** (427 cities) and **City** (1 city). All pollutant values and the AQI are complete.
- The four pollutant columns are **AQI values** of single pollutants (not raw concentrations), so they are on the same 0 - 500 scale as the overall AQI.
- The `... AQI Category` columns are text labels (Good, Moderate, ...) that are **derived from** the AQI numbers.

## Step 3.2: A Closer Look at the Output (AQI)

Before choosing an evaluation measure, it is important to look at the **range of the output**. Most cities have a moderate AQI, but a small number of cities have very high values, up to the maximum of 500.

RMSE squares each error, so these few extreme cities can make the RMSE of a fold larger than for the typical city.

In [ ]:
aqi = raw["AQI Value"]

print("Output attribute: AQI Value")
print("===========================")
print(f"Minimum : {aqi.min()}")
print(f"Median  : {aqi.median():.0f}")
print(f"Mean    : {aqi.mean():.2f}")
print(f"Maximum : {aqi.max()}")
print(f"Share of cities with AQI above 150 (Unhealthy or worse): {(aqi > 150).mean() * 100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(aqi, bins=60, color="#4C72B0")
axes[0].set(title="Distribution of AQI Value", xlabel="AQI Value", ylabel="Number of cities")
raw["AQI Category"].value_counts().reindex(
    ["Good", "Moderate", "Unhealthy for Sensitive Groups", "Unhealthy", "Very Unhealthy", "Hazardous"]
).plot.barh(ax=axes[1], color="#55A868")
axes[1].invert_yaxis()
axes[1].set(title="Cities per AQI Category", xlabel="Number of cities", ylabel="")
plt.tight_layout()
plt.show()

## Step 3.3: How Is the Overall AQI Related to the Pollutant Values?

This is the most important thing to understand about this dataset. The overall AQI is **not** an independent measurement: by the standard AQI definition it is the **highest** of the AQI values of the individual pollutants. The cell below checks this on our data.

In [ ]:
sub_index_columns = ["CO AQI Value", "Ozone AQI Value", "NO2 AQI Value", "PM2.5 AQI Value"]

highest_sub_index = raw[sub_index_columns].max(axis=1)
matches_max = (highest_sub_index == raw["AQI Value"])

print(f"AQI Value equals the HIGHEST of the 4 pollutant values in {matches_max.mean() * 100:.2f}% of cities "
      f"({(~matches_max).sum()} exceptions)")

print("\nWhich pollutant is the highest one in each city?")
which_highest = raw[sub_index_columns].idxmax(axis=1).value_counts(normalize=True) * 100
print(which_highest.round(1).to_string())

print("\nCorrelation of each pollutant AQI value with the overall AQI Value:")
print(raw[sub_index_columns + ["AQI Value"]].corr()["AQI Value"].drop("AQI Value").round(3).to_string())

**What this tells us:**

- In about **99.4%** of cities the AQI Value is exactly the **highest** of the four pollutant values (only 141 exceptions out of 23,463). This is how the AQI is defined: the worst pollutant decides the overall AQI. The exceptions are a few extreme cities with very high PM2.5, where the AQI is even higher than every pollutant value (examples are printed in Step 13).
- **PM2.5** is the highest pollutant (and therefore equals the AQI) in about **78%** of cities, **Ozone** in about **22%**, while **CO and NO2 are never the highest**. The correlation between PM2.5 and the AQI is **0.98**.
- So the AQI is not a mystery to be discovered from scratch: with PM2.5 available, a near-exact rule (*"take the maximum"*) is hidden in the data. We therefore **leave PM2.5 out of the main experiment**, as explained below.

### Design decision: PM2.5 is left out of the inputs

If PM2.5 were an input, the algorithms would only have to learn *"take the largest of these numbers"*. That is a tiny, almost fixed rule, so every reasonable algorithm would score very well and the comparison would say little about how good each algorithm is at **learning**. 

So the main experiment uses **only CO, Ozone, NO2 and Country**. This also matches a realistic use case: *estimating a city's AQI when its PM2.5 sensor is not working.*

The remaining inputs still carry a real, but much weaker, signal. This is why the problem is still learnable:

| Input | Correlation with the overall AQI |
|---|---|
| CO AQI Value | 0.43 |
| Ozone AQI Value | 0.41 |
| NO2 AQI Value | 0.23 |
| *(PM2.5 AQI Value, not used)* | *0.98* |

- Ozone is the highest pollutant in about 22% of cities. In those cities the AQI simply equals the Ozone value, which the model can use.
- In the other 78% of cities the AQI equals the hidden PM2.5 value. CO, NO2 and the country can only give indirect clues (heavily polluted places tend to be high in several pollutants, and countries differ in their typical pollution), so the model has to *estimate*, and it will make larger errors.
- The country alone explains about 46% of the variation in AQI (measured on the same data, so slightly optimistic).

The results of the full experiment **with** PM2.5 are shown in Step 13, so the effect of this decision is visible.

## Step 3.4: Pre-process the Data

Machine learning algorithms need clean numbers, so we prepare the data:

1. **Remove incomplete rows.** The 428 cities with a missing value (almost all without a country name) are removed. We still keep more than 98% of the data, so nothing important is lost, and no imputation is needed.
2. **Check for duplicates.**
3. **Drop attributes that must not be used as inputs:**
   - **City**: a different name for almost every row, so it carries no general pattern.
   - **All `... AQI Category` columns**: they are text labels created *from* the AQI numbers. Using `AQI Category` to predict `AQI Value` would be **data leakage** (giving the model the answer).

In [ ]:
"""
Pre-processing steps:
---------------------
1. Remove every row that has a missing value (these are cities without a country name).
2. Check for duplicate rows.
3. Drop attributes that must NOT be used as inputs:
   - City           : a unique name for (almost) every row, so it carries no general pattern
   - *AQI Category* : text labels derived from the AQI numbers (using them would leak the answer)
"""

rows_before = len(raw)
data = raw.dropna().reset_index(drop=True)
print(f"Rows before removing incomplete rows : {rows_before:,}")
print(f"Rows after removing incomplete rows  : {len(data):,}")
print(f"Rows removed                         : {rows_before - len(data):,}")
print(f"Duplicate rows                       : {data.duplicated().sum()}")

columns_to_drop = ["City", "AQI Category", "CO AQI Category", "Ozone AQI Category",
                   "NO2 AQI Category", "PM2.5 AQI Category"]
data = data.drop(columns=columns_to_drop)

print(f"\nAttributes kept: {list(data.columns)}")
print(f"Number of different countries: {data['Country'].nunique()}")
data.head()

# Step 4: Feature Extraction

We separate the data into the **inputs (X)** and the **output (y)**:

- **X**: the CO, Ozone and NO2 AQI values and the Country (**PM2.5 is left out**, see Step 3.3).
- **y**: the overall AQI Value that we want to predict.

In [ ]:
# PM2.5 is deliberately NOT used as an input (see Step 3.3): with it, the problem is almost a lookup.
numeric_features = ["CO AQI Value", "Ozone AQI Value", "NO2 AQI Value"]
categorical_features = ["Country"]
target = "AQI Value"

X = data[numeric_features + categorical_features]   # input (features)
y = data[target]                                    # output (what we predict)

print(f"X (inputs) : {X.shape[0]:,} rows x {X.shape[1]} columns  -> {list(X.columns)}")
print(f"y (output) : {y.shape[0]:,} values, AQI from {y.min()} to {y.max()}")
X.head()

# Step 5: Build the Pre-processing Pipeline

Algorithms such as Ridge, Lasso and SVR are sensitive to the scale of the inputs, and no algorithm can use the **text** of a country name directly. The pre-processor does two things:

| Column type | Method | Why |
|---|---|---|
| **Numerical** (3 pollutant AQI values) | `StandardScaler` | Puts all values on the same scale (mean 0, standard deviation 1) |
| **Categorical** (Country) | `OneHotEncoder` | Turns each country into its own 0/1 column; `handle_unknown="ignore"` lets the model accept a country it has not seen |

The pre-processor and the algorithm are joined in one **Pipeline**, so the same steps are applied automatically during cross validation and when the saved model is used for new cities. Inside cross validation, the scaler is learned **only on the training folds**, so no information leaks from the test fold.

In [ ]:
def make_preprocessor(numeric_cols, categorical_cols):
    """Scale the numerical inputs and one-hot encode the country."""
    return ColumnTransformer([
        ("numeric", StandardScaler(), numeric_cols),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ])

preprocessor = make_preprocessor(numeric_features, categorical_features)

# Show what the pre-processor produces
encoded = preprocessor.fit_transform(X)
print(f"Before pre-processing: {X.shape[1]} columns")
print(f"After  pre-processing: {encoded.shape[1]} columns "
      f"({len(numeric_features)} scaled numbers + {encoded.shape[1] - len(numeric_features)} country columns)")

# Step 6: Define the 5 Regression Algorithms

The five algorithms below are **different from the ones used in the Car Price notebooks** (Linear Regression, K-Nearest Neighbors, Decision Tree, Random Forest and Gradient Boosting).

| # | Algorithm | Family | Idea |
|---|---|---|---|
| 1 | **Ridge Regression** | Linear | A straight-line formula, with a penalty that keeps the weights small |
| 2 | **Lasso Regression** | Linear | A straight-line formula, with a penalty that pushes unimportant weights to exactly zero |
| 3 | **Support Vector Regression (SVR)** | Kernel | Fits a smooth curve (RBF kernel) that keeps most points inside a tolerance band |
| 4 | **Extra Trees** | Tree ensemble | Many decision trees with random split points, averaged together |
| 5 | **Histogram Gradient Boosting** | Boosted trees | Trees built one after another, each one correcting the errors of the previous ones |

The settings (hyper-parameters) are fixed and reasonable defaults; the aim of the assignment is to **compare** algorithms, not to tune each one. For SVR we use `C=100` and `epsilon=1.0` because AQI values are on a 0 - 500 scale (SVR with its default `C=1` is far too restricted for this range). SVR is therefore the only algorithm whose settings were adjusted away from the defaults, so its result should be read with that in mind.

In [ ]:
def make_algorithms():
    """Return a fresh dictionary with the 5 regression algorithms."""
    return {
        "Ridge Regression":   Ridge(alpha=1.0),
        "Lasso Regression":   Lasso(alpha=0.1),
        "Support Vector (SVR)": SVR(kernel="rbf", C=100, epsilon=1.0),
        "Extra Trees":        ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
    }

algorithms = make_algorithms()
for name, algorithm in algorithms.items():
    print(f"{name:<24} {algorithm}")

# Step 7: Set Up 10-Fold Cross Validation
## Step 7.1: Create the K-Fold Splitter

- `n_splits=10`: split the data into 10 folds
- `shuffle=True`: shuffle the cities first, so each fold is a random mix of countries
- `random_state=42`: makes the shuffle repeatable, so every algorithm is tested on **exactly the same folds**

In [ ]:
K = 10
kfold = KFold(n_splits=K, shuffle=True, random_state=42)

print(f"{K}-Fold Cross Validation created (shuffle=True, random_state=42)")

## Step 7.2: Inspect the Folds

Let us check how many cities are used for training and testing in each round.

In [ ]:
fold_sizes = []
for fold_number, (train_index, test_index) in enumerate(kfold.split(X), start=1):
    fold_sizes.append({
        "Fold (Round)": fold_number,
        "Training Cities": len(train_index),
        "Testing Cities": len(test_index),
        "Avg AQI of Test Cities": y.iloc[test_index].mean(),
        "Highest AQI in Test Fold": y.iloc[test_index].max(),
    })

fold_sizes_table = pd.DataFrame(fold_sizes).set_index("Fold (Round)")
fold_sizes_table.round(1)

Every fold has about 2,300 test cities, and the average AQI of the test cities is similar in all folds (see the table), so no single fold is dramatically easier or harder than the others.

# Step 8: Run 10-Fold Cross Validation for All 5 Algorithms

For each algorithm we build a pipeline (**pre-processing -> algorithm**). Then, for each of the 10 folds, we:

1. make a **fresh copy** of the pipeline (`clone`), so nothing learned in one round is carried into the next,
2. **train** it on the training folds,
3. **predict** the AQI of the cities in the test fold,
4. calculate **RMSE, MAE and R2** for that fold.

### Progress Bar (tqdm)

`tqdm` shows a **progress bar** while a loop runs. It is already installed in Google Colab. There is one step for each algorithm on each fold, so the bar has **5 x 10 = 50 steps**:

```
Extra Trees, fold 2/10:  64%|██████    | 32/50 [05:35<03:20, 12.1s/fold]
```

| Part of the bar | Meaning |
|---|---|
| `Extra Trees, fold 2/10` | The algorithm and fold being trained right now |
| `64%` and `32/50` | How much of the work is done (steps done / total steps) |
| `05:35` (before `<`) | **Elapsed time** (minutes:seconds) |
| `03:20` (after `<`) | **Remaining time** (estimated, minutes:seconds) |
| `12.1s/fold` | Speed |

> The remaining time is an **estimate** based on the average speed so far. Ridge and Lasso finish in a blink, while SVR and Extra Trees are much slower, so the estimate goes up and down.
>
> ⏱ **This step takes several minutes in Colab.**

In [ ]:
def run_cross_validation(numeric_cols, categorical_cols, title):
    """10-fold cross validation of all 5 algorithms using the given input columns."""
    X_cv = data[numeric_cols + categorical_cols]
    folds = list(kfold.split(X_cv))                 # the same folds for every algorithm
    algos = make_algorithms()
    results = {"rmse": {}, "mae": {}, "r2": {}, "time": {}}
    total_steps = len(algos) * K

    print(title)
    with tqdm(total=total_steps, unit="fold") as progress_bar:
        for name, algorithm in algos.items():
            model = Pipeline([
                ("preprocessor", make_preprocessor(numeric_cols, categorical_cols)),
                ("regressor", algorithm),
            ])

            rmse_scores, mae_scores, r2_scores = [], [], []
            start = time.time()
            for fold_number, (train_index, test_index) in enumerate(folds, start=1):
                progress_bar.set_description(f"{name}, fold {fold_number}/{K}")

                fold_model = clone(model)                                    # fresh, untrained copy
                fold_model.fit(X_cv.iloc[train_index], y.iloc[train_index])  # train on K-1 folds
                predictions = fold_model.predict(X_cv.iloc[test_index])      # test on 1 fold

                rmse_scores.append(np.sqrt(mean_squared_error(y.iloc[test_index], predictions)))
                mae_scores.append(mean_absolute_error(y.iloc[test_index], predictions))
                r2_scores.append(r2_score(y.iloc[test_index], predictions))
                progress_bar.update(1)

            results["time"][name] = time.time() - start
            results["rmse"][name] = np.array(rmse_scores)
            results["mae"][name] = np.array(mae_scores)
            results["r2"][name] = np.array(r2_scores)
            tqdm.write(f"{name:<24} -> mean RMSE = {results['rmse'][name].mean():>7.3f}   ({results['time'][name]:.1f} s)")

        progress_bar.set_description("Done")

    print(f"\n{K}-fold cross validation completed for all {len(algos)} algorithms.")
    return results


results = run_cross_validation(numeric_features, categorical_features,
                               "Main experiment: CO, Ozone, NO2 and Country (PM2.5 left out)")
fold_rmse = results["rmse"]

# Step 9: Compare the Algorithms
## Step 9.1: RMSE of Each Algorithm on Each Fold

Each row is one round of cross validation, and each column is one algorithm. The **lowest RMSE in each row** (the best algorithm on that fold) is highlighted in green.

**How to read it:** count the green cells in each column to see how many folds each algorithm **won**. The algorithm with the best *average* does not always win every fold, which is why we compare the mean over all folds (Step 9.2) rather than trusting any single fold.

In [ ]:
per_fold_table = pd.DataFrame(results["rmse"])
per_fold_table.index = [f"Fold {i}" for i in range(1, K + 1)]
per_fold_table.index.name = "Fold"

(per_fold_table.style
    .format("{:.3f}")
    .highlight_min(axis=1, props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))

## Step 9.2: Summary Comparison Table

This is the main result of the notebook. For each algorithm:

| Column | Meaning |
|---|---|
| **Mean RMSE** | Average RMSE over the 10 folds. This is the **main measure for ranking** the algorithms (lower is better) |
| **Std RMSE** | Standard deviation over the folds. A **small value means the algorithm is consistent** |
| **Best / Worst Fold RMSE** | Lowest and highest RMSE across the folds |
| **Mean MAE** | Average size of the error, less sensitive to a few extreme cities than RMSE |
| **Mean R2** | Share of the variation in AQI explained by the model (1.0 = perfect) |
| **Time (s)** | Total time to train and test across all 10 folds |
| **RMSE as % of Avg AQI** | Mean RMSE divided by the average AQI (about 72); puts the error in context |
| **Rank** | 1 = lowest mean RMSE = best algorithm |

In [ ]:
def make_summary(res, average_aqi):
    table = pd.DataFrame({
        "Mean RMSE": {n: v.mean() for n, v in res["rmse"].items()},
        "Std RMSE": {n: v.std() for n, v in res["rmse"].items()},
        "Best Fold RMSE": {n: v.min() for n, v in res["rmse"].items()},
        "Worst Fold RMSE": {n: v.max() for n, v in res["rmse"].items()},
        "Mean MAE": {n: v.mean() for n, v in res["mae"].items()},
        "Mean R2": {n: v.mean() for n, v in res["r2"].items()},
        "Time (s)": res["time"],
    })
    table["RMSE as % of Avg AQI"] = table["Mean RMSE"] / average_aqi * 100
    table = table.sort_values("Mean RMSE")
    table["Rank"] = range(1, len(table) + 1)
    return table

average_aqi = y.mean()
summary = make_summary(results, average_aqi)

(summary.style
    .format({"Mean RMSE": "{:.3f}", "Std RMSE": "{:.3f}", "Best Fold RMSE": "{:.3f}",
             "Worst Fold RMSE": "{:.3f}", "Mean MAE": "{:.3f}", "Mean R2": "{:.4f}",
             "Time (s)": "{:.1f}", "RMSE as % of Avg AQI": "{:.1f}%"})
    .highlight_min(subset=["Mean RMSE", "Std RMSE", "Mean MAE"],
                   props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))

## Step 9.3: Visual Comparison

**Left:** the mean RMSE of each algorithm; the black line on each bar is the standard deviation over the folds (the best algorithm is green). **Right:** the RMSE on each of the 10 folds, which shows how stable each algorithm is.

In [ ]:
def plot_comparison(res, summary_table, title):
    order = summary_table.index.tolist()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    # Left: mean RMSE with standard deviation as error bars
    means = summary_table["Mean RMSE"].values
    stds = summary_table["Std RMSE"].values
    colors = ["#2E7D4F" if i == 0 else "#4C72B0" for i in range(len(order))]
    axes[0].barh(order[::-1], means[::-1], xerr=stds[::-1], color=colors[::-1], capsize=3)
    for i, value in enumerate(means[::-1]):
        axes[0].text(value + stds[::-1][i] + 0.1, i, f"{value:.2f}", va="center", fontsize=9)
    axes[0].set(title="Mean RMSE over 10 folds (lower is better)", xlabel="RMSE (AQI points)")

    # Right: RMSE of every algorithm on every fold
    for name in order:
        axes[1].plot(range(1, K + 1), res["rmse"][name], marker="o", label=name)
    axes[1].set(title="RMSE on each fold", xlabel="Fold", ylabel="RMSE (AQI points)")
    axes[1].set_xticks(range(1, K + 1))
    axes[1].legend(frameon=False, fontsize=8)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_comparison(results, summary, "Main experiment: PM2.5 left out")

## Step 9.4: Observations

> The numbers below come from our run. If your library versions are different, your numbers may differ slightly, but the overall pattern should be the same.

1. **Histogram Gradient Boosting is the best algorithm**, with a mean RMSE of about **29.5 AQI points** (standard deviation about 0.77) and an R2 of about 0.72. It has the lowest RMSE on **9 of the 10 folds** (Extra Trees wins fold 5 by a very small margin), and it is also the fastest of the non-linear algorithms (about 25 seconds for all 10 folds).
2. **The errors are large, and that is expected.** An RMSE of about 29.5 is about 41% of the average AQI (about 72), and the typical error (MAE) is about 15 AQI points. This is the price of leaving PM2.5 out: in about 78% of cities the AQI *is* the PM2.5 value, which the model cannot see. Many cities also share exactly the same CO, Ozone, NO2 and Country values but have different AQI values (3,100 such input combinations in the data), so even a perfect model would still make errors of roughly 25 AQI points on those cities. The results are therefore a **rough estimate of the AQI, not an exact reading**.
3. **The top three algorithms are close.** Gradient Boosting (about 29.5), Extra Trees (about 31.2) and SVR (about 31.6) are within about 2 AQI points of each other, similar to the fold-to-fold standard deviation (about 0.8 to 1.7). Gradient Boosting's lead is consistent (it wins nearly every fold), but Extra Trees and SVR cannot be told apart.
4. **The linear algorithms are clearly the weakest.** Ridge (about 36.6, R2 about 0.58) and Lasso (about 37.9, R2 about 0.55) are 5 to 8 AQI points worse than the three non-linear algorithms. A straight-line formula cannot capture interactions such as *"if Ozone is high it decides the AQI, otherwise the typical pollution level of the country matters"*. Lasso is slightly worse than Ridge.
5. **RMSE and MAE do not give exactly the same ranking.** SVR has the lowest MAE (about 13.8) and Extra Trees the second lowest (about 14.5), both lower than Gradient Boosting (about 15.2). They make smaller errors on the typical city, but they make some larger mistakes, and RMSE punishes large mistakes more strongly, so Gradient Boosting ranks first on RMSE. The assignment uses RMSE, but it is useful to know that the choice of measure matters.
6. **Consistency:** Gradient Boosting also has the smallest standard deviation (about 0.77), so its score barely changes from fold to fold. SVR (about 1.70), Ridge (about 1.84) and Lasso (about 1.99) vary more.
7. **Speed:** Ridge and Lasso take under a second, Gradient Boosting about 25 seconds, Extra Trees about 4 minutes and SVR about 7 minutes (on one CPU core). Gradient Boosting is both the most accurate and one of the cheapest to train here.

## Step 9.5: Do the Algorithms Beat Simple Rules?

A model is only useful if it does better than a **trivial rule that needs no learning**. Two such rules are tested on exactly the same 10 folds:

1. **Always predict the average AQI** (the training cities' average, about 72). A model that cannot beat this has learned nothing.
2. **Take the largest of the visible values** (CO, Ozone, NO2). This is the *"pick the largest number"* rule from Step 3.3, applied to the pollutants that the model is allowed to see. It is the rule a sceptic might think the model is just following.

In [ ]:
# Two simple "rules" that need no learning, evaluated on the SAME 10 folds as the algorithms
visible_max = data[numeric_features].max(axis=1)      # largest of the 3 visible pollutant values

mean_rmse, mean_r2, rule_rmse, rule_r2 = [], [], [], []
for train_index, test_index in kfold.split(X):
    y_test = y.iloc[test_index]
    always_mean = np.full(len(test_index), y.iloc[train_index].mean())   # average AQI of the training cities
    rule = visible_max.iloc[test_index]                                   # largest visible value
    mean_rmse.append(np.sqrt(mean_squared_error(y_test, always_mean)));  mean_r2.append(r2_score(y_test, always_mean))
    rule_rmse.append(np.sqrt(mean_squared_error(y_test, rule)));         rule_r2.append(r2_score(y_test, rule))

reference = pd.DataFrame({
    "Mean RMSE": {
        "Simple rule: always predict the average AQI": np.mean(mean_rmse),
        "Simple rule: largest of CO, Ozone, NO2": np.mean(rule_rmse),
        **{name: summary.loc[name, "Mean RMSE"] for name in summary.index},
    },
    "Mean R2": {
        "Simple rule: always predict the average AQI": np.mean(mean_r2),
        "Simple rule: largest of CO, Ozone, NO2": np.mean(rule_r2),
        **{name: summary.loc[name, "Mean R2"] for name in summary.index},
    },
}).sort_values("Mean RMSE")

(reference.style
    .format({"Mean RMSE": "{:.3f}", "Mean R2": "{:.4f}"})
    .highlight_min(subset=["Mean RMSE"], props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))

**What we see:**

- **Always predicting the average AQI** gives an RMSE of about 56.3 and an R2 of about 0, as expected: it explains nothing.
- **The rule "largest of the visible values"** is even worse (RMSE about 62.5, R2 about -0.23, i.e. worse than guessing the average). In about 78% of cities the AQI is the hidden PM2.5 value, which is usually much larger than the visible pollutant values, so this rule usually predicts too low.
- **All five algorithms beat both rules.** Gradient Boosting's error is about 48% lower than that of the average-guess rule (29.5 against 56.3), and even the weakest algorithm (Lasso, about 37.9) is about 33% lower.

So the models are **not just picking the largest number**: that rule fails badly when PM2.5 is hidden. The algorithms have to combine the three pollutant values and the country in a more complex way, which is what the comparison above measures.

# Step 10: Train the Final Model
## Step 10.1: Select the Best Algorithm

**Cross validation is used to *choose* an algorithm, not to produce the final model.** The 10 models built during cross validation were only used to measure performance and are now thrown away.

We select the algorithm with the **lowest mean RMSE** from the summary table.

In [ ]:
best_algorithm_name = summary.index[0]
best_mean_rmse = summary.loc[best_algorithm_name, "Mean RMSE"]
best_std_rmse = summary.loc[best_algorithm_name, "Std RMSE"]

print(f"Best algorithm : {best_algorithm_name}")
print(f"Expected RMSE  : {best_mean_rmse:.3f} AQI points (+/- {best_std_rmse:.3f})")

## Step 10.2: Train the Best Algorithm on All the Data

Now that we know which algorithm works best, we train it on **all 23,035 cities**. There is no separate test set here, because the cross-validation RMSE from Step 9 is already our estimate of how well this model will perform on new, unseen cities.

In [ ]:
final_model = Pipeline([
    ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
    ("regressor", make_algorithms()[best_algorithm_name]),
])

final_model.fit(X, y)

print(f"Final {best_algorithm_name} model trained on {len(X):,} cities.")

## Step 10.3: Save the Trained Model

The saved file contains the **whole pipeline** (scaler, one-hot encoder **and** the algorithm), so it can predict directly from raw pollutant values and a country name. A small `model_info.json` file stores the valid input ranges and the list of countries; the web app uses it to build its input form.

To download the files to your computer in Colab, set `DOWNLOAD_MODEL = True` and run the cell again, or use the **Files** panel (folder icon) on the left.

In [ ]:
MODEL_FILE = "aqi_prediction_model.pkl"
INFO_FILE = "model_info.json"
DOWNLOAD_MODEL = False

with open(MODEL_FILE, "wb") as f:
    pickle.dump(final_model, f)

# A small description of the model, used by the web app
model_info = {
    "model": f"{best_algorithm_name} (scikit-learn Pipeline with StandardScaler + OneHotEncoder)",
    "dataset": "Global Air Pollution Dataset",
    "rows_used": int(len(X)),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "input_ranges": {c: [int(X[c].min()), int(X[c].max())] for c in numeric_features},
    "countries": sorted(X["Country"].unique().tolist()),
    "output": "AQI Value",
    "cv_folds": K,
    "cv_mean_rmse": round(float(best_mean_rmse), 4),
    "cv_std_rmse": round(float(best_std_rmse), 4),
    "cv_mean_mae": round(float(summary.loc[best_algorithm_name, "Mean MAE"]), 4),
    "cv_mean_r2": round(float(summary.loc[best_algorithm_name, "Mean R2"]), 4),
    "scikit_learn_version": sklearn.__version__,
}
with open(INFO_FILE, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

print(f"Trained model saved as '{MODEL_FILE}' ({os.path.getsize(MODEL_FILE):,} bytes)")
print(f"Model description saved as '{INFO_FILE}'")

if IN_COLAB and DOWNLOAD_MODEL:
    files.download(MODEL_FILE)
    files.download(INFO_FILE)

# Step 11: Execute the Application Phase
## Step 11.1: Take Input from User

By default the notebook uses the example city below, so that **Run all** works without stopping. To type in your own city, set `USE_INTERACTIVE_INPUT = True` and run the cell again.

In [ ]:
USE_INTERACTIVE_INPUT = False

example_city = {
    "Country": "Germany",
    "CO AQI Value": 1,
    "Ozone AQI Value": 30,
    "NO2 AQI Value": 2,
}

def get_city_input():
    print("Enter the pollutant AQI values of the city (ranges seen in the data are shown):")
    for column in numeric_features:
        print(f"  {column:<16}: {X[column].min()} - {X[column].max()}")

    city = {}
    print("\nSome of the available countries:", sorted(X["Country"].unique())[:15], "...")
    city["Country"] = input("Country: ").strip()
    for column in numeric_features:
        city[column] = int(input(f"{column}: "))
    return city

user_input = get_city_input() if USE_INTERACTIVE_INPUT else example_city

print("City air quality readings:")
for feature, value in user_input.items():
    print(f"  {feature:<16}: {value}")

## Step 11.2: Convert User Input into Feature Vector

The feature vector must have **exactly the same columns, in the same order,** as the training data. There is no need to encode it manually: the saved pipeline scales and one-hot encodes it automatically.

In [ ]:
feature_vector = pd.DataFrame([user_input])[numeric_features + categorical_features]
feature_vector

## Step 11.3: Load the Saved Model

In [ ]:
with open(MODEL_FILE, "rb") as f:
    loaded_model = pickle.load(f)

print(f"Model loaded from '{MODEL_FILE}': {loaded_model.named_steps['regressor'].__class__.__name__}")

## Step 11.4: Model Prediction

Apply the model to the feature vector of the unseen city and return the prediction to the user. The prediction is limited to the valid AQI range of 0 to 500.

In [ ]:
predicted_aqi = float(np.clip(loaded_model.predict(feature_vector)[0], 0, 500))   # AQI is always between 0 and 500

def aqi_category(value):
    if value <= 50:  return "Good"
    if value <= 100: return "Moderate"
    if value <= 150: return "Unhealthy for Sensitive Groups"
    if value <= 200: return "Unhealthy"
    if value <= 300: return "Very Unhealthy"
    return "Hazardous"

print(f"Predicted AQI Value : {predicted_aqi:.1f}  ({aqi_category(predicted_aqi)})")
print(f"Typical error (cross-validation RMSE of {best_algorithm_name}): about {best_mean_rmse:.1f} AQI points")

# Step 12: Execute the Feedback Phase

The model is deployed as a **Streamlit web app** (see the link at the top of this notebook), which includes a short feedback form. After deployment, feedback from users and domain experts (for example environmental officers) is used to improve the model. Potential improvements for this system:

1. **Use more informative inputs**: add the PM2.5 reading whenever it is available (Step 13 shows how much it helps), and add raw pollutant concentrations, weather, season or the city's population, so that the model is not limited to the pollutant AQI values.
2. **Tune hyper-parameters**: for example the number of trees in Extra Trees or the learning rate of Gradient Boosting, using cross validation (`GridSearchCV`).
3. **Add more countries' cities and more years of data**: the dataset is a single snapshot, so it cannot capture daily or seasonal changes.
4. **Predict the AQI category as well**: a classification model could directly output Good / Moderate / Unhealthy for the public.
5. **Handle very polluted cities**: a few cities have an AQI close to 500; a separate model, or an error measure less sensitive to extremes (MAE), could be reported alongside RMSE.

# Step 13: Extra Experiment: The Same Comparison WITH PM2.5

In the main experiment we left PM2.5 out (Step 3.3). Here we add it back and run the **same 10-fold cross validation, with the same 5 algorithms and the same folds**, but with **CO, Ozone, NO2, PM2.5 and Country** as inputs.

This answers two questions:

1. How much does PM2.5 help, and does it change which algorithm wins?
2. Is a model that has PM2.5 really *learning*, or just picking the largest number? (To check, we compare it with the plain rule *"AQI = the largest of the 4 values"*, which needs no learning.)

In [ ]:
all_numeric = ["CO AQI Value", "Ozone AQI Value", "NO2 AQI Value", "PM2.5 AQI Value"]

results_all = run_cross_validation(all_numeric, categorical_features,
                                   "Extra experiment: all inputs, WITH PM2.5 (CO, Ozone, NO2, PM2.5 and Country)")
summary_all = make_summary(results_all, average_aqi)

(summary_all.style
    .format({"Mean RMSE": "{:.3f}", "Std RMSE": "{:.3f}", "Best Fold RMSE": "{:.3f}",
             "Worst Fold RMSE": "{:.3f}", "Mean MAE": "{:.3f}", "Mean R2": "{:.4f}",
             "Time (s)": "{:.1f}", "RMSE as % of Avg AQI": "{:.1f}%"})
    .highlight_min(subset=["Mean RMSE", "Std RMSE", "Mean MAE"],
                   props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))

**The plain rule "AQI = the largest of the 4 pollutant values" (no learning):**

In [ ]:
# The plain rule "AQI = the largest of the 4 pollutant values" needs no learning at all
highest_of_four = data[all_numeric].max(axis=1)

rule4_rmse, rule4_mae, rule4_r2 = [], [], []
for train_index, test_index in kfold.split(data):
    y_test = y.iloc[test_index]
    rule4_rmse.append(np.sqrt(mean_squared_error(y_test, highest_of_four.iloc[test_index])))
    rule4_mae.append(mean_absolute_error(y_test, highest_of_four.iloc[test_index]))
    rule4_r2.append(r2_score(y_test, highest_of_four.iloc[test_index]))

exact = (highest_of_four == y)
print(f"Plain rule 'largest of the 4 values' : mean RMSE = {np.mean(rule4_rmse):.3f}, "
      f"mean MAE = {np.mean(rule4_mae):.3f}, mean R2 = {np.mean(rule4_r2):.4f}")
print(f"The rule is exactly right for {exact.mean() * 100:.2f}% of cities, but wrong for {(~exact).sum()} cities "
      f"(largest miss: {int((y - highest_of_four).abs().max())} AQI points).")

print("\nA few cities where the rule fails:")
data.loc[~exact, all_numeric + [target]].head(6)

**Comparison of the two experiments:**

In [ ]:
plot_comparison(results_all, summary_all, "Extra experiment: with PM2.5")

side_by_side = pd.DataFrame({
    "RMSE main (no PM2.5)": summary["Mean RMSE"],
    "RMSE with PM2.5": summary_all["Mean RMSE"],
    "R2 main (no PM2.5)": summary["Mean R2"],
    "R2 with PM2.5": summary_all["Mean R2"],
})
side_by_side["Error shrinks by (x times)"] = side_by_side["RMSE main (no PM2.5)"] / side_by_side["RMSE with PM2.5"]
side_by_side.sort_values("RMSE main (no PM2.5)").round(3)

**Observations (Step 13):**

> The numbers of the extra experiment (with PM2.5) come from our run; your Colab numbers may differ slightly in the last decimals.

1. **PM2.5 changes everything.** With it, the best RMSE falls from about 29.5 to about **2.16** AQI points (Extra Trees) and R2 rises from about 0.72 to about 0.998. PM2.5 decides the AQI in about 78% of cities, so knowing it removes most of the uncertainty.
2. **The ranking changes.** With PM2.5 the order is Extra Trees (about 2.16), SVR (about 4.33), Gradient Boosting (about 4.44), Ridge (about 8.48), Lasso (about 8.82). Without PM2.5 it is Gradient Boosting (about 29.5), Extra Trees (about 31.2), SVR (about 31.6), Ridge (about 36.6), Lasso (about 37.9). Extra Trees falls from first to second and its error grows about 14 times, more than for any other algorithm. A plausible reason (not proven here) is that with PM2.5 its fully grown trees can reproduce the near-exact rule, while without PM2.5 the remaining signal is weak and noisy, and fully grown trees fit that noise more than Gradient Boosting does.
3. **Is a model with PM2.5 just picking the largest number?** The plain rule *"largest of the 4 values"* has an RMSE of about **5.03** and is exactly right for 99.4% of cities, but it misses by up to 166 AQI points in 141 cities with extreme PM2.5. Extra Trees (about 2.16), SVR (about 4.33) and Gradient Boosting (about 4.44) all beat this rule, so they learn more than the plain maximum. Ridge and Lasso (about 8.5 to 8.8) cannot even match it, because a straight-line formula cannot represent a maximum. Still, with PM2.5 available the problem is largely a known formula, which is why it is only an **extra** experiment here and the main comparison leaves PM2.5 out.
4. **The linear algorithms lose least in relative terms** (their error grows only about 4 times, against 7 to 14 times for the non-linear algorithms), because they had no way to exploit the maximum rule in the first place.

# Conclusion

In this notebook we compared **5 regression algorithms** (Ridge, Lasso, Support Vector Regression, Extra Trees and Histogram Gradient Boosting) on the **full global air pollution dataset (23,035 cities after removing 428 incomplete rows)** using **10-fold cross validation** and **RMSE**, to predict the AQI of a city from its CO, Ozone and NO2 AQI values and its country.

- **PM2.5 was deliberately left out** of the main experiment. The overall AQI is, by definition, the highest pollutant value (99.4% of cities), so with PM2.5 the task would be almost *"pick the largest number"*. Leaving it out turns the task into estimating the AQI of a city whose PM2.5 sensor is unavailable, which is a genuine learning problem.
- **Histogram Gradient Boosting performed best** with a mean RMSE of about **29.5 AQI points** (R2 about 0.72), the lowest RMSE on 9 of 10 folds, and the shortest training time among the non-linear algorithms. Extra Trees (about 31.2) and SVR (about 31.6) followed closely, while **Ridge (about 36.6) and Lasso (about 37.9) were the weakest**. The best model was trained on all cities, saved, and used to predict the AQI of a new city.
- **All five algorithms clearly beat the simple rules** (always predicting the average AQI: about 56.3; taking the largest visible value: about 62.5), so the models are not just picking the largest number.
- **In the extra experiment with PM2.5**, the best RMSE falls to about **2.16** (Extra Trees), the order of the algorithms changes, and the models beat even the plain "largest of the four" rule (about 5.03), which shows how much one very informative input can change both the accuracy and the winner.
- **Cross validation made the comparison trustworthy.** Looking at 10 folds, not one split, showed for example that the three best algorithms are close, that Gradient Boosting is the most consistent, and that RMSE and MAE do not rank the algorithms in exactly the same order.

### Limitations

- **The predictions are rough.** The typical error is about 15 AQI points (MAE) and about 29.5 (RMSE), because the most informative pollutant (PM2.5) is hidden and many cities share identical inputs but have different AQI values.
- The dataset is a single snapshot with only pollutant AQI values and the country; real air quality also depends on weather, season and time of day.
- The hyper-parameters were fixed, not tuned (only SVR's `C` and `epsilon` were adjusted so that it can work on the 0 - 500 scale); a tuned model might change the order of the closely ranked algorithms.
- Country is encoded with one column per country, so countries with few cities (the median is 28) give the model little to learn from, and predictions for them are less reliable.